In [ ]:
# | default_exp features.endmotif_genomewide

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import numpy as np
import structlog
from kreview.eval_engine import FeatureEvaluator, parse_array

log = structlog.get_logger()

In [ ]:
# | export


class EndMotifGenomewideEvaluator(FeatureEvaluator):
    """Extracts 4-mer fragment end motif frequencies for genomewide regions.

    Produces raw 256 4-mer frequencies plus derived summary metrics:
    - Shannon entropy (cleavage site diversity)
    - DNASE1L3 signature score (CC-ending motif sum)
    - Top-10 motif concentration
    - Purine/pyrimidine asymmetry at terminal base
    """

    name = "EndMotifGenomewide"
    source_file = ".EndMotif.parquet"
    tier = 3
    category = "motifs"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            prefix = "endmotif_gw"

            if "Motif" in cols and "Frequency" in cols:
                for row in df.to_dict("records"):
                    m = str(row["Motif"])
                    if pd.notna(row["Frequency"]):
                        extracted[f"{prefix}_{m}"] = float(row["Frequency"])

            # --- Derived metrics ---
            motif_freqs = [v for k, v in extracted.items() if k.startswith(prefix)]
            if motif_freqs:
                arr = np.array(motif_freqs)
                pos = arr[arr > 0]
                if len(pos) > 0:
                    extracted[f"{prefix}_shannon_entropy"] = float(
                        -np.sum(pos * np.log2(pos))
                    )

                cc_sum = sum(
                    v
                    for k, v in extracted.items()
                    if k.startswith(prefix)
                    and len(k) > len(prefix) + 1
                    and k.endswith("CC")
                )
                if cc_sum > 0:
                    extracted[f"{prefix}_dnase1l3_score"] = float(cc_sum)

                if len(motif_freqs) >= 10:
                    top10 = sorted(motif_freqs, reverse=True)[:10]
                    extracted[f"{prefix}_top10_concentration"] = float(sum(top10))

                purine_end = sum(
                    v
                    for k, v in extracted.items()
                    if k.startswith(f"{prefix}_")
                    and len(k) == len(prefix) + 5
                    and k[-1] in ("A", "G")
                )
                pyrimidine_end = sum(
                    v
                    for k, v in extracted.items()
                    if k.startswith(f"{prefix}_")
                    and len(k) == len(prefix) + 5
                    and k[-1] in ("C", "T")
                )
                if pyrimidine_end > 0:
                    extracted[f"{prefix}_purine_pyrimidine_ratio"] = float(
                        purine_end / pyrimidine_end
                    )

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
evaluator = EndMotifGenomewideEvaluator()
df = pd.DataFrame([{"Motif": "AAAA", "Frequency": 0.05}])
res = evaluator.extract(df)
assert res["endmotif_gw_AAAA"] == 0.05